# Predicting Heart Disease Risk Using Regularized Regression Models

Heart disease is one of the leading causes of death worldwide, so being able to flag high-risk patients early actually matters. In this notebook we build and compare four different logistic regression models on the UCI Heart Disease dataset to see which one does the best job at that.

**Models we're comparing:**
- Logistic Regression (plain baseline, no regularization)
- Ridge (L2) - penalizes large coefficients
- Lasso (L1) - same idea, but also pushes some coefficients to zero (built-in feature selection)
- Polynomial + Ridge - adds interaction terms to catch non-linear patterns

**How we judge them:** cross-validated AUC-ROC, accuracy, precision, recall, confusion matrices, and feature coefficient plots.

## 1. Install & Import Dependencies

In [ ]:
# Colab already has most of these, but just to be safe
!pip install -q scikit-learn pandas numpy matplotlib seaborn

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    roc_auc_score, roc_curve, confusion_matrix, classification_report
)

import warnings
warnings.filterwarnings("ignore")

print("All good, let's go!")

## 2. Load the Dataset

Grabbing the UCI Heart Disease dataset straight from OpenML - no manual downloading needed. The target column originally has severity values from 0 to 4, but we're simplifying that down to a clean binary: **0 = no disease, 1 = disease present**.

In [ ]:
print("Pulling the dataset, hang tight...")

dataset = fetch_openml(data_id=53, as_frame=True, parser="auto")

df_full    = dataset.frame
target_col = dataset.target_names[0] if dataset.target_names else "num"
X_raw = df_full.drop(columns=[target_col]).copy()
y_raw = df_full[target_col].copy()

# OpenML can store the target as a plain number, a float string, or a full categorical.
# This handles all three so it doesn't blow up depending on the sklearn version.
if hasattr(y_raw, "cat"):
    cats = y_raw.cat.categories
    try:
        num_cats = pd.to_numeric(cats, errors="raise")
        y = y_raw.map(dict(zip(cats, num_cats.astype(int)))).astype(float)
    except (ValueError, TypeError):
        y = y_raw.cat.codes.astype(float)
        y[y == -1] = np.nan
else:
    y = pd.to_numeric(y_raw.astype(str).str.strip(), errors="coerce")

# Drop rows where the target still couldn't be read
mask = y.notna()
X = X_raw[mask].reset_index(drop=True)
y = y[mask].reset_index(drop=True).astype(int)

# Simplify to binary: sick or not sick
y = (y > 0).astype(int)

# Make sure every feature is numeric
X = X.apply(pd.to_numeric, errors="coerce")

print(f"Got it! Dataset shape: {X.shape}")
print(f"\nClass breakdown:")
print(y.value_counts().rename({0: "No Disease", 1: "Disease"}))

## 3. Exploratory Data Analysis

Before touching any models, let's actually look at what we're working with - distributions, missing values, and which features are correlated with each other.

In [ ]:
print("Quick look at the features:")
X.describe().round(2)

In [ ]:
missing = X.isnull().sum()
if missing.any():
    print("Found some missing values:")
    print(missing[missing > 0])
else:
    print("No missing values - we're good!")

In [ ]:
df = X.copy()
df["target"] = y

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart - hardcode positions as [0, 1] or matplotlib gets weird with pandas index types
counts = y.value_counts().sort_index()
axes[0].bar([0, 1], counts.values, color=["steelblue", "tomato"], edgecolor="black", width=0.5)
axes[0].set_xticks([0, 1])
axes[0].set_xticklabels(["No Disease", "Disease"])
axes[0].set_ylabel("Count")
axes[0].set_title("Class Distribution", fontsize=13)

# Correlation heatmap
corr = df.corr(numeric_only=True)
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm",
            ax=axes[1], linewidths=0.5, annot_kws={"size": 7})
axes[1].set_title("Feature Correlation Heatmap", fontsize=13)

plt.tight_layout()
plt.show()

## 4. Preprocessing

Three steps here: fill any missing values with column medians, do an 80/20 stratified split (stratified so both splits keep the same class ratio), then scale everything to zero mean and unit variance. Scaling matters a lot here because regularization penalizes coefficient size, and that penalty is unfair if features are on wildly different scales.

In [ ]:
# Fill missing values with each column's median
X = X.fillna(X.median(numeric_only=True))
feature_names = X.columns.tolist()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print(f"Train : {X_train_scaled.shape[0]} samples")
print(f"Test  : {X_test_scaled.shape[0]} samples")
print("Preprocessing done!")

## 5. Define the Models

Each model lives inside a Pipeline so everything stays tidy. For models that have a regularization strength C, we search over a range of values and let cross-validation pick the winner.

In [ ]:
cv     = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
C_grid = [0.001, 0.01, 0.1, 1, 10, 100]

model_configs = {
    "Logistic Regression": {
        "pipeline": Pipeline([
            ("clf", LogisticRegression(penalty=None, solver="lbfgs", max_iter=1000, random_state=42))
        ]),
        "params": {},   # nothing to tune on the baseline
    },
    "Ridge (L2)": {
        "pipeline": Pipeline([
            ("clf", LogisticRegression(penalty="l2", solver="lbfgs", max_iter=1000, random_state=42))
        ]),
        "params": {"clf__C": C_grid},
    },
    "Lasso (L1)": {
        "pipeline": Pipeline([
            ("clf", LogisticRegression(penalty="l1", solver="liblinear", max_iter=1000, random_state=42))
        ]),
        "params": {"clf__C": C_grid},
    },
    "Polynomial + Ridge": {
        "pipeline": Pipeline([
            ("poly", PolynomialFeatures(degree=2, include_bias=False)),
            ("clf",  LogisticRegression(penalty="l2", solver="lbfgs", max_iter=2000, random_state=42))
        ]),
        "params": {"clf__C": C_grid},
    },
}

print("All four models set up and ready to go!")

## 6. Train and Tune

GridSearchCV tries every C value in our grid and picks the one with the best AUC-ROC across 5 folds. The baseline has nothing to tune so it just gets a straight CV score.

In [ ]:
trained = {}

for name, cfg in model_configs.items():
    print(f"Training {name}...")
    if cfg["params"]:
        gs = GridSearchCV(
            cfg["pipeline"], cfg["params"],
            cv=cv, scoring="roc_auc", n_jobs=-1
        )
        gs.fit(X_train_scaled, y_train)
        trained[name] = gs.best_estimator_
        print(f"  Best C      : {gs.best_params_}")
        print(f"  CV AUC-ROC  : {gs.best_score_:.4f}")
    else:
        cfg["pipeline"].fit(X_train_scaled, y_train)
        trained[name] = cfg["pipeline"]
        cv_auc = cross_val_score(
            cfg["pipeline"], X_train_scaled, y_train,
            cv=cv, scoring="roc_auc"
        ).mean()
        print(f"  CV AUC-ROC  : {cv_auc:.4f}")
    print()

print("Done training all models!")

## 7. Test Set Evaluation

Now the real test - how do they actually perform on data they've never seen? This is where we find out if the models actually learned something useful or just memorized the training set.

In [ ]:
results = {}

for name, model in trained.items():
    y_pred = model.predict(X_test_scaled)
    y_prob = model.predict_proba(X_test_scaled)[:, 1]

    acc  = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec  = recall_score(y_test, y_pred)
    auc  = roc_auc_score(y_test, y_prob)

    results[name] = {"Accuracy": acc, "Precision": prec, "Recall": rec, "AUC-ROC": auc}

    print(f"{name}")
    print(f"  Accuracy  : {acc:.4f}")
    print(f"  Precision : {prec:.4f}")
    print(f"  Recall    : {rec:.4f}")
    print(f"  AUC-ROC   : {auc:.4f}")
    print(classification_report(y_test, y_pred, target_names=["No Disease", "Disease"]))

In [ ]:
# clean summary table - green highlights the best score in each column
results_df = pd.DataFrame(results).T.round(4)
results_df.style.highlight_max(axis=0, color="lightgreen")

## 8. ROC Curves

ROC curves show the trade-off between catching true positives and accidentally flagging healthy patients. The closer a curve hugs the top-left corner, the better the model is. The dashed line is what a random guess would look like.

In [ ]:
plt.figure(figsize=(8, 6))

for name, model in trained.items():
    y_prob = model.predict_proba(X_test_scaled)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    auc = roc_auc_score(y_test, y_prob)
    plt.plot(fpr, tpr, label=f"{name} (AUC = {auc:.3f})")

plt.plot([0, 1], [0, 1], "k--", linewidth=1, label="Random classifier")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curves - All Models")
plt.legend(loc="lower right")
plt.tight_layout()
plt.show()

## 9. Confusion Matrices

Breaking down exactly what each model got right and wrong. In a medical setting, false negatives (missing a sick patient) are way more costly than false positives, so recall matters a lot here.

In [ ]:
n = len(trained)
fig, axes = plt.subplots(1, n, figsize=(5 * n, 4))

if n == 1:
    axes = [axes]

for ax, (name, model) in zip(axes, trained.items()):
    y_pred = model.predict(X_test_scaled)
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax,
                xticklabels=["No Disease", "Disease"],
                yticklabels=["No Disease", "Disease"])
    ax.set_title(name)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")

plt.tight_layout()
plt.show()

## 10. Metrics Comparison

Side-by-side view of all four metrics across all models - easy way to see who's winning overall.

In [ ]:
results_df.plot(kind="bar", figsize=(10, 5), edgecolor="black", colormap="Set2")
plt.title("Model Performance Comparison")
plt.ylabel("Score")
plt.ylim(0.5, 1.0)
plt.xticks(rotation=20, ha="right")
plt.legend(loc="lower right")
plt.tight_layout()
plt.show()

## 11. Feature Importance

Looking at the learned coefficients to see which clinical features actually drive the predictions. Blue bars push toward a disease diagnosis, red bars push away from it.

Polynomial + Ridge is left out here - at degree 2 it generates 91 expanded features and the plot just becomes unreadable.

In [ ]:
interpretable  = ["Logistic Regression", "Ridge (L2)", "Lasso (L1)"]
models_to_plot = {k: v for k, v in trained.items() if k in interpretable}

n = len(models_to_plot)
fig, axes = plt.subplots(1, n, figsize=(6 * n, 5), sharey=True)

if n == 1:
    axes = [axes]

for ax, (name, model) in zip(axes, models_to_plot.items()):
    clf   = model.named_steps["clf"]
    coefs = clf.coef_[0]

    importance_df = pd.DataFrame({
        "Feature": feature_names,
        "Coefficient": coefs
    }).sort_values("Coefficient")

    colors = ["tomato" if c < 0 else "steelblue" for c in importance_df["Coefficient"]]
    ax.barh(importance_df["Feature"], importance_df["Coefficient"], color=colors, edgecolor="black")
    ax.axvline(0, color="black", linewidth=0.8)
    ax.set_title(f"{name}\nFeature Coefficients")
    ax.set_xlabel("Coefficient Value")

plt.tight_layout()
plt.show()

## 12. Final Summary

Let's see who won.

In [ ]:
best_auc = max(results, key=lambda k: results[k]["AUC-ROC"])
best_recall = max(results, key=lambda k: results[k]["Recall"])

print("=" * 50)
print(f"Best AUC-ROC : {best_auc} ({results[best_auc]['AUC-ROC']:.4f})")
print(f"Best Recall  : {best_recall} ({results[best_recall]['Recall']:.4f})")
print("=" * 50)
print("Recall matters most for a medical screening task (missing a sick")
print("patient is far worse than a false alarm), so despite Ridge having")
print("the highest AUC-ROC, the model with much better recall is the more")
print("practical choice here.")

print("\nFull breakdown:")
results_df.sort_values("AUC-ROC", ascending=False)